In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import os
BASE_PATH = '/content/drive/MyDrive/Colab_Notebooks/crowd-enVent'
SAVE_PATH = '/content/drive/MyDrive/2026-1/NLP/appraisal/nlp-appraisal/data'


In [3]:
import pandas as pd
gen_df = pd.read_csv(BASE_PATH + '/corpus/crowd-enVent_generation.tsv', sep='\t')

In [ ]:
# Specifying which columns I want to predict
target_dims = [
    'goal_relevance', 'self_responsblt', 'other_responsblt', 'chance_responsblt',
    'goal_support', 'predict_conseq', 'urgency', 'self_control', 'other_control',
    'chance_control', 'accept_conseq', 'social_norms', 'standards', 'attention', 'effort'
]

X = gen_df['generated_text']
y = gen_df[target_dims].copy()
emotion = gen_df['emotion'] # Keep as a reference column for later

# Normalization to [0, 1] for each target dimension
y[target_dims] = (y[target_dims] - 1) / 4

print(X.head())
print(y.head())

0    people get under my skin. Like for example if ...
1    I was driving on the highway and someone cut m...
2    someone misunderstands me and takes it out of ...
3    a child called me a name behind my back after ...
4    I was driving home from work and an individual...
Name: generated_text, dtype: object
   goal_relevance  self_responsblt  other_responsblt  chance_responsblt  \
0            0.25             0.00              1.00               0.00   
1            0.75             0.25              1.00               0.50   
2            0.50             0.25              0.75               0.25   
3            0.50             0.25              1.00               0.00   
4            0.75             0.00              1.00               0.50   

   goal_support  predict_conseq  urgency  self_control  other_control  \
0          0.50            0.75     0.25          0.75           0.00   
1          0.25            0.25     1.00          0.50           0.75   
2          0.50    

In [27]:
from sklearn.model_selection import train_test_split

# 70/15/15 split with stratification on emotion so all 13 emotion categories are represented proportionally
X_train_temp, X_test, y_train_temp, y_test, emotion_train_temp, emotion_test= train_test_split(X, y, emotion, test_size=0.15, random_state=42, stratify=emotion)
X_train, X_val, y_train, y_val, emotion_train, emotion_val = train_test_split(X_train_temp, y_train_temp, emotion_train_temp, test_size=0.1765, random_state=42, stratify=emotion_train_temp)  

print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")
print("Emotion distribution in training set:")
print(emotion_train.value_counts())
print("Emotion distribution in validation set:")
print(emotion_val.value_counts())
print("Emotion distribution in test set:")
print(emotion_test.value_counts())  

Training set size: 4619
Validation set size: 991
Test set size: 990
Emotion distribution in training set:
emotion
surprise      385
disgust       385
no-emotion    385
fear          385
boredom       385
trust         385
relief        385
sadness       385
anger         385
joy           384
pride         384
guilt         193
shame         193
Name: count, dtype: int64
Emotion distribution in validation set:
emotion
pride         83
joy           83
trust         83
anger         83
sadness       83
surprise      83
no-emotion    83
boredom       82
fear          82
disgust       82
relief        82
guilt         41
shame         41
Name: count, dtype: int64
Emotion distribution in test set:
emotion
disgust       83
boredom       83
pride         83
fear          83
relief        83
joy           83
no-emotion    82
trust         82
anger         82
surprise      82
sadness       82
guilt         41
shame         41
Name: count, dtype: int64


In [29]:
# Save the splits to CSV files immediately
dfs = {
    'train': pd.concat([X_train, y_train, emotion_train], axis=1),
    'val': pd.concat([X_val, y_val, emotion_val], axis=1),
    'test': pd.concat([X_test, y_test, emotion_test], axis=1)
}

for name, df in dfs.items(): 
    path = os.path.join(SAVE_PATH, f"{name}.csv")
    if not os.path.isfile(path):
        df.to_csv(path, index=False)
        print(f"{path} saved with {len(df)} samples.")
    else: 
        df.to_csv(path, mode='w', index=False)
        print(f"{path} overwritten with {len(df)} samples.")

/content/drive/MyDrive/2026-1/NLP/appraisal/nlp-appraisal/data/train.csv overwritten with 4619 samples.
/content/drive/MyDrive/2026-1/NLP/appraisal/nlp-appraisal/data/val.csv overwritten with 991 samples.
/content/drive/MyDrive/2026-1/NLP/appraisal/nlp-appraisal/data/test.csv overwritten with 990 samples.


In [30]:
X_train = pd.read_csv(os.path.join(SAVE_PATH, 'train.csv'))
print(X_train.tail())

                                         generated_text  goal_relevance  \
4614  I read about the £80 cut to universal credit, ...            1.00   
4615                   I was rock climbing with friends            0.00   
4616  I accidentally broke my friends passenger seat...            1.00   
4617  I felt anger when spoken about in my presence ...            0.25   
4618                           My husband had an affair            1.00   

      self_responsblt  other_responsblt  chance_responsblt  goal_support  \
4614             0.00              1.00               0.00          0.00   
4615             0.75              0.75               0.00          0.75   
4616             1.00              0.00               0.00          0.00   
4617             0.75              0.75               0.25          0.50   
4618             0.00              1.00               0.00          0.00   

      predict_conseq  urgency  self_control  other_control  chance_control  \
4614          

In [67]:
import json

# Compute inverse frequency weights and store them in a dictionary
y_train_reverse_normalized = (y_train[target_dims] * 4 + 1).round().astype(int)

dim_weights = {}

for dim in target_dims:
    freq = y_train_reverse_normalized[dim].value_counts(normalize=True)
    weights = (1 / freq) / (1 / freq).sum()
    dim_weights[dim] = {int(k): float(v) for k, v in weights.to_dict().items()}
    

# Save the weights to a JSON file
weights_path = os.path.join(SAVE_PATH, 'dim_weights.json')
with open(weights_path, 'w') as f:
    json.dump(dim_weights, f)

In [71]:
weights_json = json.load(open(weights_path))
for dim, weights in weights_json.items():
    print(f"Dimension: {dim}")
    sum_weights = sum(weights.values())
    print(f"  Sum of weights: {sum_weights:.4f}")
    for rating, weight in weights.items():
        print(f"  Rating {rating}: Weight {weight:.4f}")


Dimension: goal_relevance
  Sum of weights: 1.0000
  Rating 5: Weight 0.1507
  Rating 1: Weight 0.1680
  Rating 4: Weight 0.1862
  Rating 3: Weight 0.2242
  Rating 2: Weight 0.2709
Dimension: self_responsblt
  Sum of weights: 1.0000
  Rating 1: Weight 0.0813
  Rating 5: Weight 0.1309
  Rating 4: Weight 0.2065
  Rating 3: Weight 0.2756
  Rating 2: Weight 0.3057
Dimension: other_responsblt
  Sum of weights: 1.0000
  Rating 5: Weight 0.0926
  Rating 1: Weight 0.1090
  Rating 4: Weight 0.1718
  Rating 3: Weight 0.2983
  Rating 2: Weight 0.3283
Dimension: chance_responsblt
  Sum of weights: 1.0000
  Rating 1: Weight 0.0576
  Rating 4: Weight 0.2164
  Rating 5: Weight 0.2199
  Rating 2: Weight 0.2457
  Rating 3: Weight 0.2605
Dimension: goal_support
  Sum of weights: 1.0000
  Rating 1: Weight 0.0895
  Rating 5: Weight 0.1853
  Rating 4: Weight 0.2134
  Rating 3: Weight 0.2428
  Rating 2: Weight 0.2691
Dimension: predict_conseq
  Sum of weights: 1.0000
  Rating 4: Weight 0.1544
  Rating 1: We